# 06b_analisis_errores_y_baseline_odio

## Objetivo

1. Analizar primero los falsos positivos y falsos negativos del mejor modelo supervisado de hostilidad.
2. Entrenar un baseline **experimental** de discurso de odio mediante validación cruzada estratificada repetida.
3. Aplicar el modelo experimental al corpus formal sin presentar sus predicciones como prevalencia definitiva.

El notebook es local: no llama a la API de X ni descarga recursos.


## 1. Entradas y salidas

### Entradas
- `reports/formal_eda/manual_review_sample.csv`
- `reports/formal_ml/baseline_predictions_test.csv`
- `data/processed/x_media_anchored_interactions_corpus_formal_with_baseline_predictions.csv`

### Salidas
- diagnósticos de errores en `reports/formal_ml/`
- métricas CV experimentales de odio en `reports/formal_ml/`
- modelo en `models/formal/experimental_hate_logreg_tfidf.joblib`
- corpus ampliado en `data/processed/x_media_anchored_interactions_corpus_formal_with_hostility_and_experimental_hate_predictions.csv`


In [ ]:
import importlib
import json
import os
import re
import sys
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix


def is_project_dir(path):
    return (path / "config").exists() and (path / "data").exists() and (path / "notebooks").exists()


def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if is_project_dir(candidate):
            return candidate
        child = candidate / "HateCR"
        if is_project_dir(child):
            return child
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.labels as label_utils
import src.modeling as modeling
importlib.reload(label_utils)
importlib.reload(modeling)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "formal_ml"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models" / "formal"
FORMAL_EDA_REPORTS = PROJECT_ROOT / "reports" / "formal_eda"

for directory in [REPORTS_DIR, FIGURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MANUAL_SAMPLE_PATH = FORMAL_EDA_REPORTS / "manual_review_sample.csv"
HOSTILITY_TEST_PREDICTIONS_PATH = REPORTS_DIR / "baseline_predictions_test.csv"
HOSTILITY_CORPUS_PATH = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_with_baseline_predictions.csv"
HATE_MODEL_PATH = MODELS_DIR / "experimental_hate_logreg_tfidf.joblib"
HATE_METADATA_PATH = MODELS_DIR / "experimental_hate_label_info.json"
COMBINED_CORPUS_PATH = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_with_hostility_and_experimental_hate_predictions.csv"

RANDOM_STATE = 42
HATE_CV_SPLITS = int(os.getenv("HATE_CV_SPLITS", "5"))
HATE_CV_REPEATS = int(os.getenv("HATE_CV_REPEATS", "5"))
MIN_HATE_POSITIVES_EXPERIMENTAL = int(os.getenv("MIN_HATE_POSITIVES_EXPERIMENTAL", "10"))

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("HATE_CV_SPLITS:", HATE_CV_SPLITS)
print("HATE_CV_REPEATS:", HATE_CV_REPEATS)
print("API y descargas: desactivadas")


In [ ]:
def safe_read_csv(path, name, **kwargs):
    if not path.exists():
        print(f"[WARN] Falta {name}: {path}")
        return pd.DataFrame()
    for encoding in ["utf-8-sig", "utf-8", "latin-1"]:
        try:
            frame = pd.read_csv(path, encoding=encoding, **kwargs)
            for id_column in ["tweet_id", "review_id", "source_post_id"]:
                if id_column in frame.columns:
                    frame[id_column] = frame[id_column].astype("string")
            print(f"[OK] {name}: {len(frame):,} filas")
            return frame
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("csv", b"", 0, 1, f"No se pudo leer {path}")


def normalize_text_for_model(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    value = str(text).lower().strip()
    value = re.sub(r"https?://\S+|www\.\S+", " ", value)
    value = re.sub(r"@\w+", " ", value)
    value = re.sub(r"#(\w+)", r"\1", value)
    value = unicodedata.normalize("NFD", value)
    value = "".join(character for character in value if unicodedata.category(character) != "Mn")
    return re.sub(r"\s+", " ", value).strip()


manual_raw_df = safe_read_csv(MANUAL_SAMPLE_PATH, "muestra manual formal")
hostility_test_df = safe_read_csv(HOSTILITY_TEST_PREDICTIONS_PATH, "predicciones test hostilidad")
hostility_corpus_df = safe_read_csv(HOSTILITY_CORPUS_PATH, "corpus con predicciones de hostilidad")

if manual_raw_df.empty:
    raise ValueError("No se encontró la muestra manual formal")
if hostility_test_df.empty:
    raise ValueError("Ejecuta primero el notebook 06 para generar predicciones de prueba")
if hostility_corpus_df.empty:
    raise ValueError("Ejecuta primero el notebook 06 para generar el corpus de hostilidad")

manual_prepared_df, annotation_diagnostics = label_utils.prepare_manual_annotations(
    manual_raw_df,
    strict=True,
)
manual_ready_df = manual_prepared_df[
    manual_prepared_df["annotation_ready_for_training"]
].copy()
manual_ready_df["text_model"] = manual_ready_df["text"].map(normalize_text_for_model)
manual_ready_df["categorized_lexicon_hit_count"] = pd.to_numeric(
    manual_ready_df.get("categorized_lexicon_hit_count", 0), errors="coerce"
).fillna(0)

n_hate_positive = int((manual_ready_df["y_hate_speech"] == 1).sum())
n_hate_negative = int((manual_ready_df["y_hate_speech"] == 0).sum())
CAN_RUN_HATE_EXPERIMENT = (
    n_hate_positive >= max(HATE_CV_SPLITS, MIN_HATE_POSITIVES_EXPERIMENTAL)
    and n_hate_negative >= HATE_CV_SPLITS
)

class_distribution_df = pd.DataFrame([
    {"label": 0, "meaning": "hate_not_found", "n_rows": n_hate_negative},
    {"label": 1, "meaning": "hate_found", "n_rows": n_hate_positive},
])
class_distribution_df["percentage"] = (
    100 * class_distribution_df["n_rows"] / len(manual_ready_df)
).round(2)
class_distribution_df.to_csv(REPORTS_DIR / "hate_experimental_class_distribution.csv", index=False)

print("Etiquetas listas:", len(manual_ready_df))
print("Odio positivo:", n_hate_positive)
print("Odio negativo:", n_hate_negative)
print("CAN_RUN_HATE_EXPERIMENT:", CAN_RUN_HATE_EXPERIMENT)
display(class_distribution_df)


## 2. Análisis de errores del modelo de hostilidad

Se analizan los errores del mejor modelo supervisado del notebook 06 (`LinearSVC`).
Las tablas describen patrones observables —evento, medio, nivel, presencia de lexicón y cercanía a la frontera— sin asignar automáticamente una causa semántica.


In [ ]:
hostility_errors_df, hostility_error_tables = modeling.build_hostility_error_diagnostics(
    hostility_test_df,
    text_column="text",
)

hostility_errors_path = REPORTS_DIR / "hostility_error_review.csv"
hostility_errors_df.to_csv(hostility_errors_path, index=False)

for table_name, table in hostility_error_tables.items():
    table.to_csv(REPORTS_DIR / f"hostility_errors_{table_name}.csv", index=False)

print("Errores de hostilidad:", len(hostility_errors_df))
print("Archivo de revisión:", hostility_errors_path)
display(hostility_error_tables["summary"])
display(hostility_error_tables["by_type"])


In [ ]:
error_counts = (
    hostility_errors_df["error_type"]
    .value_counts()
    .reindex(["false_positive", "false_negative"], fill_value=0)
)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(error_counts.index, error_counts.values, color=["#C56A3B", "#3F6B7A"])
ax.set_title("Errores del mejor modelo supervisado de hostilidad")
ax.set_ylabel("Número de casos")
ax.set_xlabel("")
for position, value in enumerate(error_counts.values):
    ax.text(position, value + 0.15, str(int(value)), ha="center")
fig.tight_layout()
error_figure_path = FIGURES_DIR / "hostility_error_counts.png"
fig.savefig(error_figure_path, dpi=170)
plt.show()
print("[OK]", error_figure_path)


## 3. Baseline experimental de discurso de odio

**Advertencia explícita:** solo existen 13 casos positivos. Se utiliza validación cruzada estratificada repetida (`5 folds × 5 repeticiones`) para mostrar variabilidad entre particiones. Las métricas y predicciones resultantes son exploratorias, no estimaciones definitivas ni prevalencia poblacional.


In [ ]:
hate_fold_metrics_df = pd.DataFrame()
hate_cv_summary_df = pd.DataFrame()
hate_cv_predictions_df = pd.DataFrame()

if CAN_RUN_HATE_EXPERIMENT:
    id_columns = [
        "review_id", "tweet_id", "source_type", "event_id",
        "anchor_media_handle", "hostility_relevance_normalized",
        "y_hostility", "y_hate_speech", "text",
    ]
    hate_fold_metrics_df, hate_cv_summary_df, hate_cv_predictions_df = (
        modeling.run_repeated_stratified_text_cv(
            manual_ready_df,
            text_column="text_model",
            target_column="y_hate_speech",
            id_columns=id_columns,
            lexicon_prediction_column="categorized_lexicon_hit_count",
            n_splits=HATE_CV_SPLITS,
            n_repeats=HATE_CV_REPEATS,
            random_state=RANDOM_STATE,
        )
    )

    hate_fold_metrics_df.to_csv(
        REPORTS_DIR / "hate_experimental_cv_folds.csv", index=False
    )
    hate_cv_summary_df.to_csv(
        REPORTS_DIR / "hate_experimental_cv_summary.csv", index=False
    )
    hate_cv_predictions_df.to_csv(
        REPORTS_DIR / "hate_experimental_cv_predictions.csv", index=False
    )
    display(hate_cv_summary_df)
else:
    print("[SKIP] No hay suficientes positivos para la validación cruzada experimental.")


In [ ]:
if not hate_cv_summary_df.empty:
    plot_df = hate_cv_summary_df.sort_values("f1_positive_mean")
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(plot_df["model"], plot_df["f1_positive_mean"], color="#7D4E57")
    ax.errorbar(
        plot_df["f1_positive_mean"],
        plot_df["model"],
        xerr=plot_df["f1_positive_std"].fillna(0),
        fmt="none",
        ecolor="#2D2D2D",
        capsize=4,
    )
    ax.set_xlim(0, 1)
    ax.set_xlabel("F1 positivo medio ± desviación estándar")
    ax.set_title("Baseline experimental de odio: validación cruzada repetida")
    fig.tight_layout()
    hate_cv_figure_path = FIGURES_DIR / "hate_experimental_cv_f1.png"
    fig.savefig(hate_cv_figure_path, dpi=170)
    plt.show()
    print("[OK]", hate_cv_figure_path)

    logreg_cv_df = hate_cv_predictions_df[
        hate_cv_predictions_df["model"].eq("logreg_tfidf")
    ].copy()
    hate_cm = confusion_matrix(
        logreg_cv_df["y_true"],
        logreg_cv_df["cv_consensus_pred"],
        labels=[0, 1],
    )
    hate_cm_normalized = np.divide(
        hate_cm,
        hate_cm.sum(axis=1, keepdims=True),
        out=np.zeros_like(hate_cm, dtype=float),
        where=hate_cm.sum(axis=1, keepdims=True) != 0,
    )
    tn, fp, fn, tp = hate_cm.ravel()
    hate_cm_table_df = pd.DataFrame(
        hate_cm,
        index=["manual_sin_odio", "manual_odio"],
        columns=["pred_sin_odio", "pred_odio"],
    )
    hate_cm_table_path = REPORTS_DIR / "confusion_hate_experimental_cv_counts.csv"
    hate_cm_table_df.to_csv(hate_cm_table_path, index_label="actual_label")

    consensus_accuracy = (tn + tp) / hate_cm.sum()
    consensus_precision = tp / (tp + fp) if (tp + fp) else 0.0
    consensus_recall = tp / (tp + fn) if (tp + fn) else 0.0
    consensus_f1 = (
        2 * consensus_precision * consensus_recall / (consensus_precision + consensus_recall)
        if (consensus_precision + consensus_recall) else 0.0
    )
    consensus_specificity = tn / (tn + fp) if (tn + fp) else 0.0
    consensus_metrics_df = pd.DataFrame([
        {"metric": "accuracy", "value": consensus_accuracy},
        {"metric": "precision_hate", "value": consensus_precision},
        {"metric": "recall_hate", "value": consensus_recall},
        {"metric": "f1_hate", "value": consensus_f1},
        {"metric": "specificity_no_hate", "value": consensus_specificity},
        {"metric": "balanced_accuracy", "value": (consensus_recall + consensus_specificity) / 2},
    ])
    consensus_metrics_path = REPORTS_DIR / "confusion_hate_experimental_cv_metrics.csv"
    consensus_metrics_df.to_csv(consensus_metrics_path, index=False)

    fig, ax = plt.subplots(figsize=(7.2, 5.8))
    image = ax.imshow(hate_cm_normalized, cmap="OrRd", vmin=0, vmax=1)
    ax.set_xticks([0, 1], labels=["Sin odio", "Odio"])
    ax.set_yticks([0, 1], labels=["Sin odio", "Odio"])
    ax.set_xlabel("Predicción consensuada fuera de muestra")
    ax.set_ylabel("Etiqueta manual")
    ax.set_title("Matriz de confusión: modelo experimental de odio\nLogistic Regression, validación cruzada repetida")
    for row in range(2):
        for column in range(2):
            cell_rate = hate_cm_normalized[row, column]
            text_color = "white" if cell_rate >= 0.55 else "#2D2D2D"
            ax.text(
                column,
                row,
                f"{int(hate_cm[row, column])}\n{cell_rate:.1%}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=14,
                fontweight="bold",
            )
    colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    colorbar.set_label("Proporción dentro de la etiqueta manual")
    fig.text(
        0.5,
        0.025,
        "n=180; solo 13 positivos manuales. Los porcentajes se normalizan por fila.",
        ha="center",
        fontsize=9,
        color="#555555",
    )
    fig.tight_layout(rect=[0, 0.07, 1, 1])
    hate_cm_path = FIGURES_DIR / "confusion_hate_experimental_cv.png"
    fig.savefig(hate_cm_path, dpi=170)
    plt.show()
    print("[OK]", hate_cm_path)
    print("[OK]", hate_cm_table_path)
    print("[OK]", consensus_metrics_path)
    display(hate_cm_table_df)
    display(consensus_metrics_df)


## 4. Modelo final experimental y aplicación al corpus

Se conserva Logistic Regression por transparencia y disponibilidad de `predict_proba`. El nombre de cada columna y archivo incluye `experimental` para impedir su confusión con una clasificación validada definitivamente.


In [ ]:
hate_model = None
hate_metadata = {}

if CAN_RUN_HATE_EXPERIMENT:
    hate_model = modeling.build_logreg_tfidf_pipeline(random_state=RANDOM_STATE)
    hate_model.fit(
        manual_ready_df["text_model"].values,
        manual_ready_df["y_hate_speech"].astype(int).values,
    )
    joblib.dump(hate_model, HATE_MODEL_PATH)

    logreg_summary = hate_cv_summary_df[
        hate_cv_summary_df["model"].eq("logreg_tfidf")
    ]
    warning = (
        "EXPERIMENTAL: entrenado con solo 13 positivos manuales de odio. "
        "Las métricas tienen alta varianza y las predicciones no estiman prevalencia."
    )
    hate_metadata = {
        "status": "experimental",
        "warning": warning,
        "target_used": "y_hate_speech_from_manual_hate_speech",
        "label_source": str(MANUAL_SAMPLE_PATH),
        "n_examples": int(len(manual_ready_df)),
        "class_distribution": {"0": n_hate_negative, "1": n_hate_positive},
        "cv": {
            "type": "RepeatedStratifiedKFold",
            "n_splits": HATE_CV_SPLITS,
            "n_repeats": HATE_CV_REPEATS,
            "random_state": RANDOM_STATE,
            "logreg_summary": (
                logreg_summary.iloc[0].to_dict() if not logreg_summary.empty else {}
            ),
        },
        "model": "experimental_hate_logreg_tfidf",
        "trained_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    HATE_METADATA_PATH.write_text(
        json.dumps(hate_metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("[OK]", HATE_MODEL_PATH)
    print("[OK]", HATE_METADATA_PATH)
else:
    print("[SKIP] No se entrenó modelo experimental de odio.")


In [ ]:
combined_corpus_df = hostility_corpus_df.copy()
hate_by_source_type_df = pd.DataFrame()
hate_by_media_df = pd.DataFrame()
top_experimental_hate_df = pd.DataFrame()

if hate_model is not None:
    if "text_norm" in combined_corpus_df.columns:
        full_text = combined_corpus_df["text_norm"].where(
            combined_corpus_df["text_norm"].notna(),
            combined_corpus_df.get("text", ""),
        )
    else:
        full_text = combined_corpus_df.get("text", pd.Series("", index=combined_corpus_df.index))
    combined_corpus_df["text_model"] = full_text.fillna("").map(normalize_text_for_model)

    hate_scores = hate_model.predict_proba(combined_corpus_df["text_model"].values)[:, 1]
    combined_corpus_df["ml_hate_speech_pred_experimental"] = (hate_scores >= 0.5).astype(int)
    combined_corpus_df["ml_hate_speech_score_experimental"] = hate_scores
    combined_corpus_df["ml_hate_model_name"] = "experimental_hate_logreg_tfidf"
    combined_corpus_df["ml_hate_experimental"] = True
    if "ml_hostility_pred" in combined_corpus_df.columns:
        combined_corpus_df["ml_hate_hostility_disagreement"] = (
            combined_corpus_df["ml_hate_speech_pred_experimental"].eq(1)
            & combined_corpus_df["ml_hostility_pred"].eq(0)
        )
    else:
        combined_corpus_df["ml_hate_hostility_disagreement"] = False
    combined_corpus_df.to_csv(COMBINED_CORPUS_PATH, index=False)

    if "source_type" in combined_corpus_df.columns:
        hate_by_source_type_df = (
            combined_corpus_df.groupby("source_type", dropna=False)
            .agg(
                total_rows=("tweet_id", "size"),
                predicted_hate=("ml_hate_speech_pred_experimental", "sum"),
                mean_hate_score=("ml_hate_speech_score_experimental", "mean"),
            )
            .reset_index()
        )
        hate_by_source_type_df["predicted_hate_pct_experimental"] = (
            100 * hate_by_source_type_df["predicted_hate"] / hate_by_source_type_df["total_rows"]
        ).round(2)
        hate_by_source_type_df.to_csv(
            REPORTS_DIR / "experimental_hate_by_source_type.csv", index=False
        )

    if "anchor_media_handle" in combined_corpus_df.columns:
        hate_by_media_df = (
            combined_corpus_df.groupby("anchor_media_handle", dropna=False)
            .agg(
                total_rows=("tweet_id", "size"),
                predicted_hate=("ml_hate_speech_pred_experimental", "sum"),
                mean_hate_score=("ml_hate_speech_score_experimental", "mean"),
            )
            .reset_index()
        )
        hate_by_media_df["predicted_hate_pct_experimental"] = (
            100 * hate_by_media_df["predicted_hate"] / hate_by_media_df["total_rows"]
        ).round(2)
        hate_by_media_df.to_csv(
            REPORTS_DIR / "experimental_hate_by_anchor_media.csv", index=False
        )

    top_columns = [
        column for column in [
            "tweet_id", "text", "source_type", "event_id", "anchor_media_handle",
            "ml_hostility_pred", "ml_hostility_score",
            "ml_hate_speech_pred_experimental", "ml_hate_speech_score_experimental",
        ] if column in combined_corpus_df.columns
    ]
    top_experimental_hate_df = combined_corpus_df.nlargest(
        50, "ml_hate_speech_score_experimental"
    )[top_columns].copy()
    top_experimental_hate_df.to_csv(
        REPORTS_DIR / "top_experimental_hate_texts.csv", index=False
    )

    disagreement_columns = [
        column for column in [
            "tweet_id", "text", "source_type", "event_id", "anchor_media_handle",
            "ml_hostility_pred", "ml_hostility_score",
            "ml_hate_speech_pred_experimental", "ml_hate_speech_score_experimental",
            "ml_hate_hostility_disagreement",
        ] if column in combined_corpus_df.columns
    ]
    hate_hostility_disagreements_df = combined_corpus_df[
        combined_corpus_df["ml_hate_hostility_disagreement"]
    ][disagreement_columns].copy()
    hate_hostility_disagreements_df.to_csv(
        REPORTS_DIR / "experimental_hate_hostility_disagreements.csv", index=False
    )

    predicted_hate_n = int(combined_corpus_df["ml_hate_speech_pred_experimental"].sum())
    predicted_hate_pct = round(100 * predicted_hate_n / len(combined_corpus_df), 2)

    hate_prediction_distribution_df = (
        combined_corpus_df["ml_hate_speech_pred_experimental"]
        .value_counts(dropna=False)
        .reindex([0, 1], fill_value=0)
        .rename_axis("prediction_value")
        .reset_index(name="n_comments")
    )
    hate_prediction_distribution_df["prediction_label"] = (
        hate_prediction_distribution_df["prediction_value"].map(
            {0: "Sin predicción de odio", 1: "Predicción experimental de odio"}
        )
    )
    hate_prediction_distribution_df["percentage"] = (
        hate_prediction_distribution_df["n_comments"]
        / hate_prediction_distribution_df["n_comments"].sum()
        * 100
    ).round(2)
    hate_distribution_path = REPORTS_DIR / "experimental_hate_prediction_distribution.csv"
    hate_prediction_distribution_df.to_csv(hate_distribution_path, index=False)

    negative_row = hate_prediction_distribution_df.iloc[0]
    positive_row = hate_prediction_distribution_df.iloc[1]
    fig, ax = plt.subplots(figsize=(11, 4.0))
    ax.barh(
        ["Corpus completo"],
        [negative_row["percentage"]],
        color="#355C68",
        height=0.46,
    )
    ax.barh(
        ["Corpus completo"],
        [positive_row["percentage"]],
        left=[negative_row["percentage"]],
        color="#8F2D2D",
        height=0.46,
    )

    negative_count_label = f"{int(negative_row['n_comments']):,}".replace(",", ".")
    positive_count_label = f"{int(positive_row['n_comments']):,}".replace(",", ".")
    ax.text(
        negative_row["percentage"] / 2,
        0,
        f"Sin predicción de odio\n{negative_count_label} ({negative_row['percentage']:.2f}%)",
        ha="center",
        va="center",
        color="white",
        fontsize=11,
        fontweight="bold",
    )
    positive_center = negative_row["percentage"] + positive_row["percentage"] / 2
    ax.annotate(
        f"Predicción experimental de odio\n{positive_count_label} ({positive_row['percentage']:.2f}%)",
        xy=(positive_center, 0.20),
        xytext=(77, 0.72),
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        color="#702020",
        arrowprops={"arrowstyle": "->", "color": "#702020", "linewidth": 1.4},
    )

    total_label = f"{len(combined_corpus_df):,}".replace(",", ".")
    ax.set_xlim(0, 100)
    ax.set_ylim(-0.62, 1.08)
    ax.set_xlabel("Porcentaje de comentarios")
    ax.set_title(
        "Modelo experimental: distribución de predicciones de discurso de odio\n"
        f"n = {total_label} comentarios"
    )
    for spine_name in ["top", "right", "left"]:
        ax.spines[spine_name].set_visible(False)
    ax.tick_params(axis="y", length=0)
    fig.text(
        0.5,
        0.025,
        "Resultado exploratorio: el modelo se entrenó con 13 positivos manuales; no estima prevalencia real.",
        ha="center",
        fontsize=9,
        color="#555555",
    )
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    hate_distribution_figure_path = FIGURES_DIR / "experimental_hate_prediction_distribution.png"
    fig.savefig(hate_distribution_figure_path, dpi=180, bbox_inches="tight")
    plt.show()

    print("Corpus ampliado:", COMBINED_CORPUS_PATH)
    print("Predicciones experimentales positivas:", predicted_hate_n)
    print("Porcentaje experimental:", predicted_hate_pct)
    print("Desacuerdos odio=1 y hostilidad=0:", len(hate_hostility_disagreements_df))
    print("Distribución experimental:", hate_distribution_path)
    print("Figura experimental:", hate_distribution_figure_path)

    if "ml_hostility_pred" in combined_corpus_df.columns:
        agreement_counts = pd.crosstab(
            combined_corpus_df["ml_hostility_pred"],
            combined_corpus_df["ml_hate_speech_pred_experimental"],
        ).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
        agreement_percent = agreement_counts / len(combined_corpus_df) * 100

        agreement_records = []
        for hostility_value in [0, 1]:
            for hate_value in [0, 1]:
                count = int(agreement_counts.loc[hostility_value, hate_value])
                agreement_records.append({
                    "hostility_prediction": hostility_value,
                    "hostility_label": "no_hostil" if hostility_value == 0 else "hostil",
                    "hate_prediction_experimental": hate_value,
                    "hate_label": "sin_odio" if hate_value == 0 else "odio",
                    "n_comments": count,
                    "percentage_corpus": round(100 * count / len(combined_corpus_df), 2),
                })
        agreement_df = pd.DataFrame(agreement_records)
        agreement_path = REPORTS_DIR / "full_corpus_hostility_hate_prediction_agreement.csv"
        agreement_df.to_csv(agreement_path, index=False)

        fig, ax = plt.subplots(figsize=(7.5, 6.0))
        image = ax.imshow(agreement_percent.values, cmap="YlGnBu", vmin=0, vmax=agreement_percent.values.max())
        ax.set_xticks([0, 1], labels=["Sin odio", "Odio"])
        ax.set_yticks([0, 1], labels=["No hostil", "Hostil"])
        ax.set_xlabel("Predicción experimental de odio")
        ax.set_ylabel("Predicción de hostilidad")
        ax.set_title("Corpus completo: coincidencia entre predicciones\nHostilidad y discurso de odio experimental")
        for row in range(2):
            for column in range(2):
                cell_pct = agreement_percent.iloc[row, column]
                text_color = "white" if cell_pct >= 30 else "#1F2933"
                ax.text(
                    column,
                    row,
                    f"{int(agreement_counts.iloc[row, column]):,}\n{cell_pct:.2f}%".replace(",", "."),
                    ha="center",
                    va="center",
                    color=text_color,
                    fontsize=14,
                    fontweight="bold",
                )
        colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        colorbar.set_label("Porcentaje del corpus")
        fig.text(
            0.5,
            0.02,
            "Matriz descriptiva entre modelos; no es una matriz de confusión ni usa verdad de referencia.",
            ha="center",
            fontsize=9,
            color="#555555",
        )
        fig.tight_layout(rect=[0, 0.07, 1, 1])
        agreement_figure_path = FIGURES_DIR / "full_corpus_hostility_hate_prediction_agreement.png"
        fig.savefig(agreement_figure_path, dpi=180, bbox_inches="tight")
        plt.show()
        print("Matriz descriptiva del corpus:", agreement_path)
        print("Figura de coincidencia:", agreement_figure_path)
    else:
        print("[SKIP] No hay predicciones de hostilidad para construir la matriz descriptiva.")
else:
    print("[SKIP] No se aplicó modelo experimental al corpus.")


## 5. Advertencias metodológicas

1. Trece positivos no permiten un estimador estable de discurso de odio.
2. La validación cruzada repetida muestra sensibilidad a distintas particiones, pero no crea información nueva.
3. `manual_hostility` y `manual_hate_speech` son targets distintos.
4. Las predicciones experimentales de odio no deben presentarse como prevalencia.
5. Los falsos positivos y negativos requieren lectura contextual del comentario y del post ancla.
6. El corpus está anclado en medios costarricenses y no representa toda la conversación política en X.
7. Antes de utilizar el modelo de odio como resultado sustantivo deben ampliarse y, si es posible, codificarse doblemente los casos positivos.


In [ ]:
run_summary_df = pd.DataFrame([
    {"metric": "manual_rows", "value": len(manual_ready_df)},
    {"metric": "manual_hate_positive", "value": n_hate_positive},
    {"metric": "manual_hate_negative", "value": n_hate_negative},
    {"metric": "hostility_test_errors", "value": len(hostility_errors_df)},
    {"metric": "hostility_false_positives", "value": int((hostility_errors_df["error_type"] == "false_positive").sum())},
    {"metric": "hostility_false_negatives", "value": int((hostility_errors_df["error_type"] == "false_negative").sum())},
    {"metric": "hate_cv_splits", "value": HATE_CV_SPLITS},
    {"metric": "hate_cv_repeats", "value": HATE_CV_REPEATS},
    {"metric": "hate_model_status", "value": "experimental" if hate_model is not None else "not_trained"},
    {"metric": "combined_corpus_rows", "value": len(combined_corpus_df) if hate_model is not None else 0},
    {"metric": "hate_hostility_prediction_disagreements", "value": int(combined_corpus_df.get("ml_hate_hostility_disagreement", pd.Series(dtype=bool)).sum()) if hate_model is not None else 0},
    {"metric": "methodological_warning", "value": "Do not interpret experimental hate predictions as prevalence"},
])
run_summary_path = REPORTS_DIR / "hate_experimental_run_summary.csv"
run_summary_df.to_csv(run_summary_path, index=False)
print("[OK]", run_summary_path)
display(run_summary_df)
